# 09. Функции потерь и метрики

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/09_losses_metrics.ipynb)

Код из раздела [modules/09_losses_metrics.md](https://github.com/justxor/math-for-ai-2026/blob/main/modules/09_losses_metrics.md#m09). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### Метрическое и контрастное обучение

In [ ]:
import torch, torch.nn.functional as F
def clip_loss(img_emb, txt_emb, tau=0.07):
    img, txt = F.normalize(img_emb, dim=-1), F.normalize(txt_emb, dim=-1)
    logits = img @ txt.T / tau                     # (B, B): диагональ — правильные пары
    labels = torch.arange(len(img))
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

### 🏋️ Практика модуля

**9.2.** Реализуйте ROC-AUC через вероятностное определение и сравните со sklearn.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
y = np.random.rand(2000) < 0.1
s = np.random.randn(2000) + 1.2 * y
pos, neg = s[y], s[~y]
auc = ((pos[:, None] > neg[None, :]).mean() + 0.5 * (pos[:, None] == neg[None, :]).mean())
print(auc, roc_auc_score(y, s))   # совпадают

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def roc_pr():
    rng = np.random.default_rng(0)
    y = np.r_[np.zeros(900), np.ones(100)]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
    for sep, c in zip([0.5, 1.5, 3.0], C):
        s = np.r_[rng.normal(0, 1, 900), rng.normal(sep, 1, 100)]
        order = np.argsort(-s); ys = y[order]
        tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
        tpr = tp / ys.sum(); fpr = fp / (1 - ys).sum(); prec = tp / (tp + fp)
        auc = np.trapezoid(tpr, fpr)
        axes[0].plot(fpr, tpr, color=c, lw=2, label=f"разделимость {sep}: AUC={auc:.2f}")
        axes[1].plot(tpr, prec, color=c, lw=2)
    axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="случайный: 0.5")
    axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR (recall)"); axes[0].legend(fontsize=9)
    axes[0].set_title("ROC-кривая")
    axes[1].axhline(0.1, color="k", ls="--", lw=1); axes[1].text(0.6, 0.13, "базовый уровень = доля позитивов (10%)", fontsize=8)
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("PR-кривая — честнее при дисбалансе")
    save(fig, "roc_pr")

roc_pr()

In [ ]:
def calibration():
    rng = np.random.default_rng(0)
    n = 20000
    true_p = rng.beta(2, 2, n); y = rng.random(n) < true_p
    logit = np.log(true_p / (1 - true_p))
    over = 1 / (1 + np.exp(-2.5 * logit))      # переуверенная модель
    fig, ax = plt.subplots(figsize=(5.8, 5))
    bins = np.linspace(0, 1, 11)
    for p, c, lab in [(true_p, C[2], "откалибрована"), (over, C[1], "переуверена (как часто у глубоких сетей)")]:
        idx = np.digitize(p, bins) - 1
        conf = [p[idx == b].mean() for b in range(10) if (idx == b).sum() > 50]
        acc = [y[idx == b].mean() for b in range(10) if (idx == b).sum() > 50]
        ece = sum((idx == b).mean() * abs(p[idx == b].mean() - y[idx == b].mean()) for b in range(10) if (idx == b).sum() > 0)
        ax.plot(conf, acc, "o-", color=c, lw=2, label=f"{lab}, ECE={ece:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="идеал")
    ax.set_xlabel("предсказанная вероятность"); ax.set_ylabel("реальная доля позитивов")
    ax.set_title("Reliability diagram"); ax.legend(fontsize=8)
    save(fig, "calibration")

calibration()